# PDEFlow · recoverable heat / wave training

**A1 representation → A2 physical evolution → B conditional generation.** Same physical-state v2 capacities; no attention redesign. Standard MSE is the default field loss; region-balanced MSE (α=.5) is experimental. Baseline FM and decoded endpoint likelihood are separate choices. Two-update pipeline checks do not assess accuracy.

Eight steps below. Training, benchmarks, downloads and diagnostics are off until explicitly enabled. A failed representation gate stops progression. CUDA/Drive execution of this revision still needs the acceptance check in `docs/observation-aligned-training.md`.


In [ ]:
#@title 1 · Set up project (archive or private/public checkout)
from pathlib import Path
import os, sys, subprocess, zipfile, hashlib, json, tempfile, urllib.parse
REPOSITORY_URL = "" #@param {type:"string"}
REVISION = "" #@param {type:"string"}
GITHUB_SECRET_NAME = "" #@param {type:"string"}
SOURCE_ARCHIVE = "" #@param {type:"string"}
CACHE_ROOT = Path("/content/pdeflow-source") # disposable; configurable
INSTALL_PROJECT = True #@param {type:"boolean"}
EXPECTED_SOURCE_SHA256 = "" # optional handoff hash, never inferred from a model
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
if not REPOSITORY_URL and not SOURCE_ARCHIVE:
    from google.colab import files
    uploaded = files.upload()  # upload the source ZIP, not model weights
    if len(uploaded) != 1: raise ValueError("Upload exactly one source ZIP")
    SOURCE_ARCHIVE = next(iter(uploaded))
if REPOSITORY_URL:
    parsed = urllib.parse.urlsplit(REPOSITORY_URL)
    if parsed.username or parsed.password or parsed.query or parsed.fragment: raise ValueError("Do not put tokens in a URL; use a runtime secret name")
    if not REVISION: raise ValueError("Set an exact checkout revision for reproducible training")
    source_key = hashlib.sha256((REPOSITORY_URL + REVISION).encode()).hexdigest()
    PACKAGE_ROOT = CACHE_ROOT / source_key[:16]
    if not PACKAGE_ROOT.exists():
        env = os.environ.copy()
        with tempfile.TemporaryDirectory() as auth:
            if GITHUB_SECRET_NAME:
                from google.colab import userdata
                token = userdata.get(GITHUB_SECRET_NAME)
                ask = Path(auth) / "askpass.py"
                ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PDEFLOW_GIT_TOKEN"])\n')
                ask.chmod(0o700); env.update(GIT_ASKPASS=str(ask), PDEFLOW_GIT_TOKEN=token, GIT_TERMINAL_PROMPT="0")
            for cmd in (["git", "clone", "--no-checkout", REPOSITORY_URL, str(PACKAGE_ROOT)], ["git", "-C", str(PACKAGE_ROOT), "checkout", "--detach", REVISION]):
                result = subprocess.run(cmd, env=env, capture_output=True, text=True)
                if result.returncode: raise RuntimeError("Checkout failed; check revision/access. Credentials are not printed.")
    revision = subprocess.check_output(["git", "-C", str(PACKAGE_ROOT), "rev-parse", "HEAD"], text=True).strip()
    print("Source revision:", revision)
else:
    archive = Path(SOURCE_ARCHIVE).expanduser().resolve()
    source_key = hashlib.sha256(archive.read_bytes()).hexdigest()
    if EXPECTED_SOURCE_SHA256 and source_key != EXPECTED_SOURCE_SHA256: raise ValueError("Unexpected source archive identity")
    extracted = CACHE_ROOT / source_key[:16]
    if not extracted.exists():
        with zipfile.ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist()) > 512 * 1024**2: raise ValueError("Source ZIP exceeds 512 MiB unpacked")
            for i in z.infolist():
                p = Path(i.filename)
                if p.is_absolute() or ".." in p.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            z.extractall(extracted)
    roots = [p.parent for p in extracted.rglob("pyproject.toml") if (p.parent / "src/pdeflow").is_dir()]
    if len(roots) != 1: raise ValueError("ZIP must contain exactly one PDEFlow source root")
    PACKAGE_ROOT = roots[0]
    # Rerun safety: verify cached members rather than trusting a stale extraction.
    with zipfile.ZipFile(archive) as z:
        for i in z.infolist():
            member = Path(i.filename)
            if member.is_absolute() or ".." in member.parts or (i.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("Unsafe archive member")
            if i.is_dir(): continue
            cached = extracted / member
            if cached.is_symlink() or not cached.resolve().is_relative_to(extracted.resolve()) or not cached.is_file() or hashlib.sha256(cached.read_bytes()).digest() != hashlib.sha256(z.read(i)).digest():
                raise ValueError("Cached source changed; choose a fresh CACHE_ROOT. Existing files are not overwritten.")
    print("Source archive SHA256:", source_key)
subprocess.run([sys.executable, str(PACKAGE_ROOT / "scripts/colab_setup.py"), *([] if INSTALL_PROJECT else ["--check-only"])], check=True)
# Editable .pth files installed by subprocess pip are not reloaded in this kernel.
if "pdeflow" in sys.modules and not Path(sys.modules["pdeflow"].__file__).resolve().is_relative_to(PACKAGE_ROOT.resolve()):
    raise RuntimeError("Source changed in this running kernel. Restart, then select the intended source again.")
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
import importlib
importlib.invalidate_caches()
# --no-deps setup retains the runtime's compatible PyTorch/CUDA stack.
# If verification fails, select a supported Python3.11/3.12 runtime (e.g. 2026.07).
# No automatic PyTorch replacement or hidden runtime restart occurs.

from pdeflow.source_bundle import verify as verify_source
if not REPOSITORY_URL: print("Verified source:", verify_source(archive, PACKAGE_ROOT))
from pdeflow.coordinate.optimization import scientific_source
print("Scientific source identity:", scientific_source()["sha256"])


## 2 · Persistent storage before training

Drive must be mounted and pass a write/read probe. Runtime disk is disposable. Saves are synchronous at safe optimizer boundaries: first valid checkpoint, successful-update cadence, stage transitions and safe stop/final. A hard termination can lose work since the last **verified** snapshot. This is not an atomic Drive rename guarantee.

New experiments receive unique IDs independent of display names. For a fresh runtime: reload the same source ZIP in step1, select a saved `RUN_ID` and `RUN_MODE="resume"` here, then run step3's restore branch. It uses recorded science, not today's preset. Set `SNAPSHOT_ID` to choose an older complete snapshot; blank scans for latest valid, even with a stale pointer. Use an empty local root if local files conflict. Do not overwrite them.


In [ ]:
#@title 2 · Mount/configure persistence and choose new versus restore
ENABLE_DRIVE_PERSISTENCE = True #@param {type:"boolean"}
DRIVE_MOUNT = "/content/drive" #@param {type:"string"}
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/PDEFlow-training" #@param {type:"string"}
LOCAL_ROOT = "/content/pdeflow-training" #@param {type:"string"}
RUN_MODE = "new" #@param ["new", "resume"]
RUN_NAME = "shared-medium-observation-control" # display name only
RUN_ID = "" # required on resume; printed for new runs
SNAPSHOT_ID = "" # optional verified snapshot selection
SAVE_EVERY_SUCCESSFUL_UPDATES = 100 #@param {type:"integer"}
MIN_SAVE_SECONDS = 0 # optional throttle; first/stage/final saves bypass it
KEEP_SNAPSHOTS = 3 # last N + pinned best/final; selected owned run only
SAVE_FAILURE_POLICY = "stop" #@param ["stop", "continue"]
# continue explicitly risks unsaved work; retries are bounded, no copy queue.
if ENABLE_DRIVE_PERSISTENCE:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    if not Path(DRIVE_MOUNT).is_mount():
        raise RuntimeError("Drive is not mounted; refusing to train against a disposable directory labelled Drive")
    from pdeflow.persistence import probe
    print("Storage probe:", probe(Path(DRIVE_RUNS_ROOT) / "probe-run"))
else:
    print("PERSISTENCE DISABLED: runtime loss can lose ALL local artifacts. Choose this only deliberately.")
Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
from pdeflow.colab import discover
from pdeflow.persistence import snapshots
print("Complete runs:", discover(DRIVE_RUNS_ROOT) if ENABLE_DRIVE_PERSISTENCE else [])
if RUN_MODE == "resume":
    if not RUN_ID: raise ValueError("Choose a saved RUN_ID")
    for r in snapshots(Path(DRIVE_RUNS_ROOT) / RUN_ID):
        print(r["id"], "sequence",r.get("sequence"),"stage",r.get("stage"),"update",r.get("successful_update"),"bytes",r["unique_bytes"])


## 3 · Choose science, or restore recorded science

Initial-field mixture types describe shapes, **not PDE families**. Heat state is u; wave state is displacement u and velocity v. Physical time t is independent of flow time s. Effective batch = examples per successful optimizer update; GPU microbatch = examples per neural pass; accumulation combines passes with per-example weighting.

The large preset remains 100k updates per stage; capacity and update budget are independent. Optional losses do not assert better accuracy. `ADVANCED` groups the scientific settings; do not alter them on resume. An inference import cannot resume training. Stage C and default correction are deferred.

Choose **Standard MSE** for the baseline or **Region-balanced MSE — experimental** for equal detected-feature emphasis. Alpha defaults to .5 only for the latter. Training loss controls learning. Display metrics only change evaluation.


In [ ]:
#@title 3 · Model, objectives / Fresh-runtime restore and resume
from pdeflow.colab import training_preset, Session
from pdeflow.runtime import read_json, write_json
PRESET = "medium" #@param ["pipeline-check", "xsmall", "small", "medium", "large", "xlarge"]
DEVICE = "cuda" #@param ["cuda", "mps", "cpu"]
PRECISION = "auto" # T4 -> native FP16 + GradScaler, no emulated BF16
FAMILIES = "joint" #@param ["joint", "heat", "wave"]
FIELD_LOSS = "mse" #@param ["mse", "region_balanced_mse", "weighted_mse", "mae", "spectral_mse"]
FEATURE_BALANCE = 0.5 # advanced; used only for region_balanced_mse
FLOW_OBJECTIVE = "baseline" #@param ["baseline", "observation-aligned"]
ALLOW_CPU_PIPELINE_CHECK = False
# Advanced: each term is independent and recorded. Model discrepancy is an
# explicit physical noise assumption relative to global quantity scales.
ADVANCED = {}
OBSERVATION_AUXILIARY = dict(beta=.01, minimum_s=.5, maximum_s=1., sensor_budget=8,
    query_chunk=8, selection="uniform", reduction="mean", noiseless_policy="tolerance",
    tolerance_absolute=.001, tolerance_relative=.05, discrepancy_absolute=0.,
    discrepancy_relative=.05, gaussian_constant=False, warmup_updates=100)
OBSERVATION_TIME = dict(policy="continuous_uniform", stratified=True) # NEW runs; each sensor has known continuous physical time
# Optional: dict(policy="continuous_mixture",whole=.65,early=.25,initial=.1)
A2_PHYSICS = dict(weight=0., query_points=4) # optional FP32 strong derivatives
SPECTRAL = dict(spectral_weight=.1 if FIELD_LOSS=="spectral_mse" else 0.,
    spectral_resolution=16, spectral_frequency_power=0.) # +256 queries/quantity
if RUN_MODE == "resume":
    session = Session.from_store(DRIVE_RUNS_ROOT, RUN_ID, LOCAL_ROOT,
        snapshot=SNAPSHOT_ID or None, device=DEVICE, allow_cpu_check=ALLOW_CPU_PIPELINE_CHECK)
    config = session.c
    print("RESTORED recorded source/config/objectives/counters, including full encoder and dependencies.")
else:
    overrides = {**ADVANCED,
        "observation_time": {**OBSERVATION_TIME,**ADVANCED.get("observation_time",{})},
        "family_weights": {"heat":.5,"wave":.5} if FAMILIES=="joint" else {FAMILIES:1.},
        "field_loss": {"type":FIELD_LOSS,**({"alpha":FEATURE_BALANCE} if FIELD_LOSS=="region_balanced_mse" else {}),**SPECTRAL,**ADVANCED.get("field_loss",{})},
        "joint": {**ADVANCED.get("joint",{}),"observation_auxiliary":OBSERVATION_AUXILIARY if FLOW_OBJECTIVE=="observation-aligned" else {}},
        "evolution": {**ADVANCED.get("evolution",{}),"physics":A2_PHYSICS}}
    config = training_preset(PRESET, DEVICE, PRECISION, overrides)
    session = Session.new(config, RUN_NAME, LOCAL_ROOT, DRIVE_RUNS_ROOT,
        allow_cpu_check=ALLOW_CPU_PIPELINE_CHECK, persistence_enabled=ENABLE_DRIVE_PERSISTENCE,
        save_updates=SAVE_EVERY_SUCCESSFUL_UPDATES, save_min_seconds=MIN_SAVE_SECONDS,
        keep_snapshots=KEEP_SNAPSHOTS, save_retries=2, save_failure=SAVE_FAILURE_POLICY)
    RUN_ID = session.name
    write_json(session.root / "source-input.json", {"archive_or_checkout_sha256":source_key})
print("RUN_ID (keep this):", session.name)
print("Fast disk:",session.root,"· Persistent:",session.persistent)
print(json.dumps(session.inspect(),indent=2))


## 4 · Optional bounded profile / batch experiment

Benchmarks use disposable clones and do not apply settings or bypass the real representation gate. Keep workload/precision fixed for throughput comparisons. Inspect cold/warm cache and endpoint-likelihood overhead separately. A short preflight is an estimate, not a learning result. Substantial training requires a matching preflight.


In [ ]:
#@title 4 · Optional benchmarks (never auto-applied)
RUN_PREFLIGHT = False
RUN_BATCH_EXPERIMENT = False
if RUN_PREFLIGHT:
    profile_result = session.preflight(updates=10, warmup=2)
    print(json.dumps(profile_result,indent=2))
if RUN_BATCH_EXPERIMENT:
    from pdeflow import batch_experiments
    batch_result = session.benchmark_batch_sizes(background=False,mode="efficiency",
        stages=["autoencoder","evolution","flow"],candidates=[8,16,32],
        effective_batch=32,warmup=2,measured=5,max_seconds=180)
    BATCH_EXPERIMENT_ID = batch_result["id"]
    print(batch_experiments.failure_details(batch_result))
    print(json.dumps(batch_result,indent=2))
# Inspect/cancel later: session.batch_experiment(ID), session.stop_batch_experiment(ID)


## 5 · Explicitly apply batch settings, inspect the resolved plan

Applying settings rebuilds the actual Session and invalidates stale preflight. Resume retains recorded budgets/settings unless you explicitly authorize continuation. `keep_examples` may reduce update budgets; incompatible warmup is rejected unless **new-run** `rescale_warmup` is explicitly selected. Learning rate is not silently scaled. Review both updates and total examples.


In [ ]:
#@title 5 · Apply / show resolved plan (does not train)
APPLY_BATCH_SETTINGS = False
APPLY_BENCHMARK_RECOMMENDATION = False
EFFECTIVE_BATCH = 8
GPU_MICROBATCH = "auto"
STAGE_MICROBATCHES = {} # e.g. {"autoencoder":8,"evolution":8,"flow":8}
BUDGET_POLICY = "keep_updates" # or keep_examples
SCHEDULE_POLICY = "reject" # explicit new-run alternative: rescale_warmup
EXPLICIT_CONTINUATION = False
if APPLY_BENCHMARK_RECOMMENDATION:
    session = session.apply_batch_result(BATCH_EXPERIMENT_ID,
        budget_policy=BUDGET_POLICY,continuation=EXPLICIT_CONTINUATION)
elif APPLY_BATCH_SETTINGS:
    session = session.apply_training_settings(dict(effective_batch=EFFECTIVE_BATCH,
        microbatch=GPU_MICROBATCH,stages=STAGE_MICROBATCHES,schedule_policy=SCHEDULE_POLICY),
        budget_policy=BUDGET_POLICY,continuation=EXPLICIT_CONTINUATION)
config = session.c
from pdeflow.coordinate.physics import training_spec
from pdeflow.coordinate.optimization import options
print("Training families:",json.dumps(training_spec(config),indent=2))
print("Actual stages / updates / examples / microbatch / accumulation:",json.dumps(session.training_summary(),indent=2))
print("Schedules:",json.dumps({s:options(config)[s] for s in ("autoencoder","evolution","flow")},indent=2))
from pdeflow.field_losses import training_summary as field_training_summary
print(field_training_summary(config)["description"])
print("Field query counts:",field_training_summary(config))
print("A2 optional physics:",config.evolution.get("physics",{}))
print("B: masked/group-balanced FM + endpoint likelihood:",config.joint["observation_auxiliary"])
print("A2 target times (independent of B):",config.interval, config.evolution.get("time_policy","legacy_query_v1"),config.time_sampling.get("query", "initial=.25 early=.35 uniform=.35 final=.05"))
print("B known measurement times:",config.observation_time)
print("Persistence:",session.persistence_enabled,session.persistent,"every",session.save_updates,
    "successful updates, minimum seconds",session.save_min_seconds,"failure policy",session.save_failure)
print("Rerun optional preflight AFTER any application. No estimate establishes accuracy.")


## 6 · Train or resume

The minimum representation gate remains enforced. A1-only/failed-gate artifacts are useful for diagnostics but are not a generative model. Stop safely with the runtime interrupt or `session.stop()` from another cell, then run step7. Periodic persistence already protects earlier completed boundaries. Do not depend on a final cell surviving runtime termination.

To extend a run, restore it first, explicitly set larger budgets with `dataclasses.replace(session.c, flow_steps=...)`, keep the saved schedule, and call `mode="extend"`. This differs from same-experiment resume and from starting a new model from weights.


In [ ]:
#@title 6 · Train / resume (off by default)
RUN_TRAINING = False
STAGES = "both" # both / representation / evolution / flow
from pdeflow.notebook import run
if RUN_TRAINING:
    outcome = run(session, stages=STAGES, mode=RUN_MODE, override_gate=False)
    print("Workflow:",outcome["state"])
else:
    print("Training off. Review resolved plan and preflight before enabling.")


## 7 · Save and verify complete run

Idempotent: no training or benchmark. Three manifests share canonical files rather than duplicating weights: **inference** (no encoder/optimizer), **diagnostic** (exact E+T+D+B and provenance), **resumable** (all available stage optimizers/schedulers/scalers/RNG/counters/dependencies and exact source). A partial run is labelled with missing stages. Repeated download ZIPs are reused by manifest hash and are excluded from periodic snapshots.


In [ ]:
#@title 7 · Save and verify complete run
SAVE_AND_VERIFY = True
DOWNLOAD_ARTIFACT = "none" # none / inference / diagnostic / resumable
if SAVE_AND_VERIFY and any((session.root/s/"resume.pt").exists() for s in ("autoencoder","evolution","flow")):
    saved = session.finalize(standalone_check=True)
    print(json.dumps(saved,indent=2))
    if DOWNLOAD_ARTIFACT != "none":
        from pdeflow.training_artifacts import download_archive
        archive_path = download_archive(session.root,DOWNLOAD_ARTIFACT)
        from google.colab import files
        files.download(str(archive_path))
else:
    print("No trained stage checkpoint yet. No generative model claimed. Source/config persist when training starts.")


## 8 · Optional diagnostics, after recovery is safe

Diagnostic loading requires the exact matching full-state encoder and frozen evolution, never an unrelated encoder. Evaluate conditioning sensors, independent held-out sensors, fields (initial/early/late), u/v and physical invariants separately. Empirical ensemble spread is not calibrated uncertainty. Never tune on the untouched test split. The next bounded ablation is described in `docs/observation-aligned-training.md`.

Back on the Mac: import `export/coordinate.pt` in the Heat/Wave playground for inference. Keep the verified diagnostic/resumable manifests and referenced objects on Drive for recovery. Reload **the saved source**, not an arbitrary newer ZIP, on resume. Source/config changes require an explicit new experiment or recorded incompatible continuation, not a silent fallback.


In [ ]:
#@title 8 · Read-only A2 export / optional diagnostics
"""Read-only diagnostic export; this function also runs in an OLD active kernel.

Only stdlib + the already-installed torch are used. No installation, source edit,
model forward, synchronization request, optimizer operation or Drive write.
"""
from pathlib import Path
import json,hashlib,tempfile,time,zipfile,platform
import torch


def export_a2_diagnostics(run_root,destination,include_weights=True,max_weight_mib=512,persistent_root=None):
    run_root=Path(run_root);destination=Path(destination)
    if destination.exists():raise FileExistsError('Choose a new diagnostic ZIP; no run file is overwritten')
    destination.parent.mkdir(parents=True,exist_ok=True)
    files={};notes=[];cp_path=run_root/'evolution/resume.pt'
    with tempfile.TemporaryDirectory(prefix='pdeflow-a2-export-') as d:
        temporary=Path(d)
        for relative in ('session.json','resolved-config.json','source-input.json','source/identity.json','source/source.zip','representation-quality.json','representation-validation.json','representation-validation.npz','evolution-validation.json','evolution-validation.npz','sync-status.json','autoencoder/run.json','evolution/run.json','evolution/config.json','evolution/history.json','evolution/history-live.jsonl','evolution/evolution-preview.json','evolution/evolution-preview.npz'):
            p=run_root/relative
            if p.is_file():
                if p.stat().st_size>100*1024**2:notes.append('Metadata file exceeds explicit100MiB export cap: '+relative);continue
                data=p.read_bytes();target=temporary/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data);files[relative]=target
        checkpoint_info=None
        if cp_path.exists():
            # Trainer replaces checkpoints atomically. mmap keeps the opened file
            # inode stable while only selected state pages are serialized.
            cp=torch.load(cp_path,map_location='cpu',weights_only=True,mmap=True)
            checkpoint_info={k:cp.get(k) for k in ('stage','progress','attempted_updates','skipped_updates','actual_precision','field_loss','resolved_field_loss','dependency','config','scientific_source','dataset_identity')}
            checkpoint_info['scheduler']=cp.get('scheduler');checkpoint_info['history']=cp.get('history',[])
            weights=cp.get('autoencoder')
            if include_weights:
                if not weights or not any(k.startswith('encoder.') for k in weights):raise ValueError('A2 snapshot lacks full matching state encoder; no substitute was made')
                size=sum(v.numel()*v.element_size() for v in weights.values())
                if size>max_weight_mib*1024**2:raise ValueError(f'A2 weights use {size/1024**2:.1f}MiB, exceeding the explicit cap; set include_weights=False or deliberately increase max_weight_mib')
                diagnostic={k:v for k,v in cp.items() if k not in ('optimizer','scheduler','scaler','rng','torch_rng','cuda_rng','ambient_rng')}
                diagnostic['artifact_role']='A2 diagnostic-only snapshot; not resumable training state'
                target=temporary/'a2-diagnostic.pt';torch.save(diagnostic,target);files[target.name]=target
                checkpoint_info['exported_weight_bytes']=size
            checkpoint_info['snapshot_semantics']='Existing complete local checkpoint, possibly older than live run.json; progress here identifies the exported weights. No new checkpoint was requested.'
        else:notes.append('No existing A2 checkpoint yet; metadata only')
        persistent=[]
        if persistent_root:
            for p in (Path(persistent_root)/'snapshots').glob('*.json'):
                try:
                    r=json.loads(p.read_text());persistent.append({'id':r['id'],'stage':r.get('stage'),'successful_update':r.get('successful_update'),'created_unix':r['created_unix'],'manifest':str(p),'verification':'Candidate metadata only; objects were NOT scanned during active training'})
                except (OSError,ValueError,KeyError):continue
        report={'run_id':run_root.name,'created_unix':time.time(),'checkpoint':checkpoint_info,'notes':notes,'persistent_snapshot_candidates':persistent,'environment':{'torch':str(torch.__version__),'python':platform.python_version(),'cuda_runtime':torch.version.cuda,'cuda_available':torch.cuda.is_available()},'safety':'Read-only old-runtime export; no model evaluation/training/source modification. Diagnostic weights omit optimizer/RNG and cannot resume training.'}
        target=temporary/'diagnostic-summary.json';target.write_text(json.dumps(report,indent=2)+'\n');files[target.name]=target
        manifest={'format':1,'files':{n:{'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for n,p in files.items()}}
        with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_STORED) as archive:
            for name,p in files.items():archive.write(p,name)
            archive.writestr('DIAGNOSTIC_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
    with zipfile.ZipFile(destination) as archive:
        if archive.testzip() is not None:raise ValueError('Diagnostic ZIP failed CRC verification')
        for name,e in manifest['files'].items():
            if hashlib.sha256(archive.read(name)).hexdigest()!=e['sha256']:raise ValueError('Diagnostic checksum mismatch: '+name)
    return {'path':str(destination),'sha256':hashlib.sha256(destination.read_bytes()).hexdigest(),'files':len(files),'checkpoint_update':checkpoint_info.get('progress') if checkpoint_info else None,'contains_diagnostic_weights':include_weights and 'a2-diagnostic.pt' in files,'not_a_resume_bundle':True}

# Copy this WHOLE cell into the existing active runtime: no pip/reload needed.
EXPORT_A2_DIAGNOSTICS = False
INCLUDE_A2_WEIGHTS = True  # matching E+T+D, capped512MiB; no optimizer
DOWNLOAD_DIAGNOSTIC_ZIP = False
if EXPORT_A2_DIAGNOSTICS:
    import uuid
    diagnostic_zip = Path(LOCAL_ROOT) / ("a2-diagnostics-" + session.name + "-" + uuid.uuid4().hex[:8] + ".zip")
    diagnostic_export = export_a2_diagnostics(session.root, diagnostic_zip, INCLUDE_A2_WEIGHTS, persistent_root=session.persistent)
    print(json.dumps(diagnostic_export,indent=2))
    if DOWNLOAD_DIAGNOSTIC_ZIP:
        from google.colab import files
        files.download(str(diagnostic_zip))

RUN_DIAGNOSTIC_LOAD = False
if RUN_DIAGNOSTIC_LOAD:
    from pdeflow.training_artifacts import load_diagnostic, verify_view
    diagnostic = load_diagnostic(session.root,device=DEVICE)
    print("Exact encoder available:",diagnostic.ae.encoder is not None)
    print(json.dumps(verify_view(session.root,"diagnostic")["stages"],indent=2))
